In [ ]:
# Mega Harness — Z-Image-Turbo API no Google Colab (uma célula)
# Ativa GPU no Colab e executa esta célula. Copia a Base URL e a chave para
# Mega Harness → Definições → Imagens. Mantém o runtime e o URL privados.
import os, sys, time, re, io, base64, secrets, shutil, subprocess, threading, urllib.request
from IPython.display import display, HTML

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "diffusers", "transformers", "accelerate", "safetensors", "fastapi", "uvicorn"], check=True)
import torch
from diffusers import ZImagePipeline
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from PIL import Image
import uvicorn

if not torch.cuda.is_available():
    raise RuntimeError("Ativa um acelerador GPU nas definições do Google Colab.")
subprocess.run(["nvidia-smi"], check=True)

# O dtype adapta-se à GPU do Colab (por exemplo, T4 usa FP16).
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("A carregar Z-Image-Turbo; a primeira execução descarrega os pesos do modelo…")
pipe = ZImagePipeline.from_pretrained(
    "Tongyi-MAI/Z-Image-Turbo",
    torch_dtype=dtype
)
pipe.to("cuda")
pipe.set_progress_bar_config(disable=False)
print("✅ Z-Image-Turbo carregado.")

API_KEY = secrets.token_urlsafe(32)
app = FastAPI(title="Mega Harness Z-Image API", docs_url=None, redoc_url=None)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["Authorization", "Content-Type"]
)

class ImageRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=4000)
    width: int = Field(default=1024, ge=512, le=1024)
    height: int = Field(default=1024, ge=512, le=1024)
    num_inference_steps: int = Field(default=8, ge=8, le=8)
    seed: int | None = None


def auth(value):
    if not value or not secrets.compare_digest(value, "Bearer " + API_KEY):
        raise HTTPException(status_code=401, detail="Chave API inválida.")

@app.get("/health")
def health(authorization: str | None = Header(default=None)):
    auth(authorization)
    return {"ok": True, "model": "Tongyi-MAI/Z-Image-Turbo"}

@app.post("/v1/images/generations")
def generate(body: ImageRequest, authorization: str | None = Header(default=None)):
    auth(authorization)
    # O modelo foi feito para oito passos; limita a resolução para caber melhor
    # nas GPUs de 16 GB disponíveis no Colab.
    width = (body.width // 16) * 16
    height = (body.height // 16) * 16
    generator = torch.Generator(device="cuda")
    if body.seed is not None:
        generator.manual_seed(body.seed)
    with torch.inference_mode():
        image = pipe(
            prompt=body.prompt,
            width=width,
            height=height,
            num_inference_steps=8,
            guidance_scale=0.0,
            generator=generator
        ).images[0]
    buffer = io.BytesIO()
    image.save(buffer, format="PNG", optimize=True)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return {"created": int(time.time()), "data": [{"b64_json": encoded, "revised_prompt": body.prompt}]}

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=7860, log_level="warning"))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
for _ in range(120):
    if server.started:
        break
    time.sleep(1)
if not server.started:
    raise RuntimeError("A API local não iniciou.")

cloudflared = shutil.which("cloudflared")
if not cloudflared:
    deb = "/content/cloudflared.deb"
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        deb
    )
    subprocess.run(["dpkg", "-i", deb], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    cloudflared = shutil.which("cloudflared")

def start_tunnel():
    log_path = "/content/cloudflared-zimage.log"
    log = open(log_path, "w", buffering=1)
    process = subprocess.Popen(
        [cloudflared, "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate"],
        stdout=log, stderr=subprocess.STDOUT, start_new_session=True
    )
    url = None
    for _ in range(90):
        if process.poll() is not None:
            break
        try:
            text = open(log_path, errors="ignore").read()
        except Exception:
            text = ""
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
        if match:
            url = match.group(0)
            break
        time.sleep(2)
    if not url:
        tail = open(log_path, errors="ignore").read()[-2500:]
        raise RuntimeError("Não consegui abrir o túnel Cloudflare:\n" + tail)
    return process, url

tunnel, public_url = start_tunnel()

def show_config():
    print("\n✅ Z-Image-Turbo ligado")
    print("Base URL:", public_url)
    print("Chave API:", API_KEY)
    print("\nMega Harness → Definições → Imagens")
    print("Mantém a Base URL e a chave privadas.")

show_config()
while True:
    if not server_thread.is_alive():
        raise RuntimeError("A API Z-Image terminou.")
    if tunnel.poll() is not None:
        print("\nO túnel parou; vou abrir outro endereço.")
        tunnel, public_url = start_tunnel()
        show_config()
    time.sleep(10)
